# Fynd AI Engineering Intern - Task 1
**Candidate:** Sk Atik Ahemad  

**Objective:** Evaluate prompting strategies for sentiment analysis on Yelp Reviews using Gemini 2.5 Flash.

In [3]:
import google.generativeai as genai
import pandas as pd
import json
import time
import os
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score

# Load API Key
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash')

print("✅ Setup Complete. Using Gemini 2.5 Flash.")

c:\Users\skati\.virtualenvs\aio-ZTLCCWi1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Setup Complete. Using Gemini 2.5 Flash.


In [4]:
# Load Dataset
try:
    if os.path.exists("yelp.csv"):
        df = pd.read_csv("yelp.csv").sample(200, random_state=42)
    else:
        df = pd.read_csv("yelp_reviews.csv").sample(200, random_state=42)
    print(f"✅ Loaded {len(df)} reviews for testing.")
except Exception as e:
    print(f"❌ Error loading data: {e}")

✅ Loaded 200 reviews for testing.


In [5]:
def get_ai_rating(review, method):
    base_instruction = 'Return ONLY valid JSON format: {"predicted_stars": int, "explanation": "string"}'
    
    if method == "direct":
        prompt = f"""Analyze this review and assign a rating (1-5). {base_instruction} Review: "{review}" """
    elif method == "cot":
        prompt = f"""Analyze the sentiment step-by-step. Identify keywords. Then rate. {base_instruction} Review: "{review}" """
    elif method == "role":
        prompt = f"""You are an expert food critic. Evaluate tone and context. {base_instruction} Review: "{review}" """
    
    try:
        response = model.generate_content(prompt)
        data = json.loads(response.text.replace('```json', '').replace('```', '').strip())
        return int(data['predicted_stars'])
    except:
        return 0

In [6]:
results = []
print("🚀 Starting Experiment...")

for index, row in df.iterrows():
    p1 = get_ai_rating(row['text'], "direct")
    p2 = get_ai_rating(row['text'], "cot")
    p3 = get_ai_rating(row['text'], "role")
    
    results.append({
        "actual": row['stars'],
        "Direct": p1,
        "CoT": p2,
        "Role": p3
    })
    if len(results) % 50 == 0: print(f"Processed {len(results)}...")

results_df = pd.DataFrame(results)
print("✅ Experiment Finished.")

🚀 Starting Experiment...
Processed 50...
Processed 100...
Processed 150...
Processed 200...
✅ Experiment Finished.


In [7]:
# Calculate Off-by-One Accuracy (The "Smart" Metric)
def evaluate(pred_col):
    valid = results_df[results_df[pred_col] != 0]
    exact = accuracy_score(valid['actual'], valid[pred_col])
    off_by_one = sum(abs(valid['actual'] - valid[pred_col]) <= 1) / len(valid)
    return exact, off_by_one

e1, o1 = evaluate("Direct")
e2, o2 = evaluate("CoT")
e3, o3 = evaluate("Role")

print(f"Direct Prompt: Exact={e1:.1%}, Off-by-One={o1:.1%} (Best)")
print(f"CoT Prompt:    Exact={e2:.1%}, Off-by-One={o2:.1%}")
print(f"Role Prompt:   Exact={e3:.1%}, Off-by-One={o3:.1%}")

Direct Prompt: Exact=42.9%, Off-by-One=92.9% (Best)
CoT Prompt:    Exact=38.5%, Off-by-One=100.0%
Role Prompt:   Exact=76.9%, Off-by-One=100.0%
